# **Cyber Threat Detection with Deep Learning (BETH Dataset)**

> **Dataset:** BETH simulated real-world Linux host system logs (preprocessed split provided by DataCamp)  
> **Task:** Binary classification : detect suspicious system-call events (`sus_label = 1`) vs. benign (`sus_label = 0`)  
> **Model:** Multi-Layer Perceptron (MLP) trained with PyTorch

## 1. Project Overview

- **Dataset Characteristics:** The BETH dataset captures real Linux host telemetry (eBPF system-call traces) enriched with honeypot attack data. Each row represents a single system-call event described by seven numerical features: `processId`, `threadId`, `parentProcessId`, `userId`, `mountNamespace`, `argsNum`, and `returnValue`. The target label `sus_label` marks an event as benign (0) or suspicious/malicious (1). The dataset is heavily class-imbalanced, with only ~0.17 % of events labelled malicious.

- **Data Preprocessing:** Features are normalised with `StandardScaler` fitted exclusively on the training split to prevent data leakage. Labels are converted to `float32` tensors compatible with `BCEWithLogitsLoss`. No imputation is required as the preprocessed splits contain no missing values.

- **Handling Class Imbalance:** Rather than resampling (SMOTE/undersampling), we pass a `pos_weight ≈ 600` to the loss function. This scalar penalises false negatives — missing a real attack — roughly 600× more than false positives.

- **Model Architecture:** A three-layer MLP (`7 → 64 → 32 → 1`) with ReLU activations and 30% Dropout is used. The output is a single logit converted to a probability via sigmoid. The model is trained with the Adam optimiser at a learning rate of 1e-4 for 14 epochs.

- **Performance Evaluation:** The model is evaluated on a held-out validation set using accuracy, precision, recall, F1-score, and a confusion matrix. Despite the severe class imbalance, the model achieves near-perfect detection with very few false negatives.

## 2. Import Libraries

In [15]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

from torchmetrics import Accuracy
import joblib

In [29]:
import torch
import numpy as np
import random

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Additional settings for GPU (CUDA) to ensure deterministic behavior
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 3. Dataset Characteristics and Exploratory Overview

### 3.1 Load Data

In [16]:
# Load CSV files (train / validation / test)
# Place the three CSV files in the same directory as this notebook
train_df = pd.read_csv('labelled_train.csv')
val_df   = pd.read_csv('labelled_validation.csv')
test_df  = pd.read_csv('labelled_test.csv')

train_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,7337,1,100,4026532231,5,0,1
1,381,7337,1,100,4026532231,1,0,1
2,381,7337,1,100,4026532231,0,0,1
3,7347,7347,7341,0,4026531840,2,-2,1
4,7347,7347,7341,0,4026531840,4,0,1


### 3.2 Dataset Size and Class Distribution

In [17]:
print(f"Training set shape : {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape      : {test_df.shape}")
print()

# Class balance in the training set
# ~99.83 % benign vs ~0.17 % malicious — a ~600:1 imbalance ratio
print("Training label distribution (%):\n",
      train_df['sus_label'].value_counts(normalize=True).mul(100).round(3))

Training set shape : (763144, 8)
Validation set shape: (188967, 8)
Test set shape      : (188967, 8)

Training label distribution (%):
 sus_label
0    99.834
1     0.166
Name: proportion, dtype: float64


### 3.3 Feature Summary

In [18]:
# Statistical summary of numerical features
train_df.describe()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
count,763144.000000,763144.000000,763144.000000,763144.000000,7.631440e+05,763144.000000,763144.000000,763144.000000
mean,6814.763308,6820.265241,1882.216609,2.279034,4.026532e+09,2.672082,17.520924,0.001663
std,1948.871187,1937.068333,2215.563094,37.416576,1.649030e+02,1.340906,318.596662,0.040744
min,1.000000,1.000000,0.000000,0.000000,4.026532e+09,0.000000,-115.000000,0.000000
25%,7313.000000,7313.000000,187.000000,0.000000,4.026532e+09,1.000000,0.000000,0.000000
50%,7365.000000,7365.000000,1385.000000,0.000000,4.026532e+09,3.000000,0.000000,0.000000
75%,7415.000000,7415.000000,1648.000000,0.000000,4.026532e+09,4.000000,4.000000,0.000000
max,8619.000000,8619.000000,7672.000000,1000.000000,4.026532e+09,5.000000,8289.000000,1.000000


## 4. Data Preprocessing

### 4.1 Feature / Label Split

In [19]:
# Separate features (X) from the binary target label (y) for each split
X_train, y_train = train_df.drop('sus_label', axis=1), train_df['sus_label']
X_val,   y_val   = val_df.drop('sus_label',   axis=1), val_df['sus_label']
X_test,  y_test  = test_df.drop('sus_label',  axis=1), test_df['sus_label']

### 4.2 Feature Normalisation

We fit `StandardScaler` **only on the training set** and apply the same transform to validation and test sets. Fitting on all data would leak distributional statistics from held-out sets into training.

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train only
X_val_scaled   = scaler.transform(X_val)          # transform only — no fitting
X_test_scaled  = scaler.transform(X_test)         # transform only — no fitting

### 4.3 Convert to PyTorch Tensors

In [21]:
# All arrays are cast to float32: required by nn.Linear and BCEWithLogitsLoss
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_scaled,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32)

y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_val_t   = torch.tensor(y_val.values,   dtype=torch.float32)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.float32)

### 4.4 DataLoaders and Positive-Class Weight

The `pos_weight` parameter scales the loss contribution of positive (malicious) samples.
With a 600:1 ratio, setting `pos_weight = 600` makes the model treat each missed attack as
600× more costly than a false alarm — appropriate for a security context where missed detections
are far more dangerous than false positives.

In [22]:
# Derived from class imbalance: 99.833 / 0.167 ≈ 600
pos_weight = torch.tensor([600.4], dtype=torch.float32)

# Batch size of 256 balances GPU/CPU utilisation with gradient noise
# shuffle=True on train to prevent learning order-dependent patterns
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=256, shuffle=False)

## 5. Model Architecture

A compact three-layer **Multi-Layer Perceptron (MLP)** is used. Each hidden layer is followed by a ReLU activation and a Dropout layer (p=0.3) to reduce overfitting. The output layer produces a single raw logit, which is converted to a probability by the loss function internally.

```
Input (7) → Linear(64) → ReLU → Dropout(0.3)
          → Linear(32) → ReLU → Dropout(0.3)
          → Linear(1)  [logit]
```

In [23]:
input_dim = X_train_t.shape[1]  # 7 features

model = nn.Sequential(
    nn.Linear(input_dim, 64),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(32, 1)             # single logit output for binary classification
)

print(model)

Sequential(
  (0): Linear(in_features=7, out_features=64, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=64, out_features=32, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=32, out_features=1, bias=True)
)


## 6. Training

### 6.1 Loss Function and Optimiser

In [24]:
# BCEWithLogitsLoss fuses sigmoid + binary cross-entropy in a single numerically stable step.
# pos_weight amplifies the gradient signal when the model misses a malicious event.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 14

### 6.2 Training Loop

In [25]:
for epoch in range(EPOCHS):

    # Training phase 
    model.train()           # enables Dropout
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                          # clear accumulated gradients
        logits = model(X_batch)                        # forward pass
        loss   = criterion(logits, y_batch.unsqueeze(1))
        loss.backward()                                # backpropagation
        optimizer.step()                               # gradient descent step
        total_loss += loss.item()

    # Validation phase 
    model.eval()            # disables Dropout
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss   = criterion(val_logits, y_val_t.unsqueeze(1))

        # Convert logits → probabilities → binary predictions (threshold = 0.5)
        val_preds = (torch.sigmoid(val_logits) > 0.5).long().squeeze()
        val_true  = y_val_t.long()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss.item():.4f}")

Epoch 01/14 | Train Loss: 0.6393 | Val Loss: 0.1014
Epoch 02/14 | Train Loss: 0.2777 | Val Loss: 0.0958
Epoch 03/14 | Train Loss: 0.2440 | Val Loss: 0.1046
Epoch 04/14 | Train Loss: 0.2273 | Val Loss: 0.1036
Epoch 05/14 | Train Loss: 0.2160 | Val Loss: 0.1037
Epoch 06/14 | Train Loss: 0.2163 | Val Loss: 0.1053
Epoch 07/14 | Train Loss: 0.1951 | Val Loss: 0.1078
Epoch 08/14 | Train Loss: 0.1910 | Val Loss: 0.1071
Epoch 09/14 | Train Loss: 0.1849 | Val Loss: 0.1057
Epoch 10/14 | Train Loss: 0.1777 | Val Loss: 0.1043
Epoch 11/14 | Train Loss: 0.1711 | Val Loss: 0.1038
Epoch 12/14 | Train Loss: 0.1623 | Val Loss: 0.1010
Epoch 13/14 | Train Loss: 0.1614 | Val Loss: 0.0928
Epoch 14/14 | Train Loss: 0.1525 | Val Loss: 0.0940


## 7. Performance Evaluation

### 7.1 Validation Accuracy

In [26]:
accuracy_metric = Accuracy(task='binary')
val_accuracy    = accuracy_metric(val_preds, val_true).item() * 100

print(f"Validation Accuracy: {val_accuracy:.2f}%")

Validation Accuracy: 99.99%


### 7.2 Classification Report and Confusion Matrix

Key metrics to focus on for an imbalanced security dataset:
- **Recall (class 1):** proportion of actual attacks correctly detected (false negatives are costly)
- **Precision (class 1):** proportion of flagged events that are genuinely malicious (false positives increase analyst workload)
- **F1-score:** balances precision and recall

In [27]:
print("Classification Report:")
print(classification_report(val_true, val_preds, target_names=['Benign (0)', 'Suspicious (1)']))

print("Confusion Matrix:")
print(confusion_matrix(val_true, val_preds))
print("\nRows = Actual | Columns = Predicted")
print("[[TN  FP]")
print(" [FN  TP]]")

Classification Report:
                precision    recall  f1-score   support

    Benign (0)       1.00      1.00      1.00    188181
Suspicious (1)       1.00      0.99      0.99       786

      accuracy                           1.00    188967
     macro avg       1.00      0.99      1.00    188967
  weighted avg       1.00      1.00      1.00    188967

Confusion Matrix:
[[188178      3]
 [     9    777]]

Rows = Actual | Columns = Predicted
[[TN  FP]
 [FN  TP]]


## 8. Save Model Artefacts

Both the trained model weights and the fitted scaler must be saved together.
At inference time, **all new inputs must pass through the same scaler** before being fed to the model,
using a different normalisation would silently degrade predictions.

In [28]:
torch.save(model.state_dict(), 'model.pth')    # model weights only
joblib.dump(scaler, 'scaler.pkl')              # scaler, must be used for all future inputs

['scaler.pkl']